# End-to-End Example: Computing Representational Geometry Markers

This notebook demonstrates the full marker-computation pipeline from the paper:

> **Diagnosing Generalization Failures from Representational Geometry Markers**  
> Chi-Ning Chou, Artem Kirsanov, Yao-Yuan Yang, SueYeon Chung — *ICLR 2026*

Starting from a trained ResNet-18 checkpoint, we:
1. Load the CIFAR-10 test set (auto-downloaded if not present)
2. Extract penultimate-layer (avgpool) features and output logits
3. Compute **logit-based markers** (AUROC, ATC score, energy score, …)
4. Compute **statistical markers** (sparsity, mean distance, covariance, …)
5. Compute **geometric markers** via GLUE (D_eff, R_eff, Ψ_eff, participation ratio)

**Expected runtime:** ~2 min on CPU (the geo analysis dominates). GPU reduces this to ~30 s.

In [1]:
import os
import sys
import numpy as np
import torch
import torchvision
import torchvision.transforms as transforms

sys.path.insert(0, os.path.abspath(''))
from analysis import logit_analysis, stat_analysis, geo_analysis
from models import build_model

In [2]:
# ── Load CIFAR-10 test set ────────────────────────────────────────────────────
# Set CIFAR10_ROOT to a directory containing the cifar-10-batches-py/ folder.
# If the data is not present, torchvision will download it (~170 MB).
CIFAR10_ROOT = './data/cifar10'  # adjust to your local path

CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD  = (0.2023, 0.1994, 0.2010)

testset = torchvision.datasets.CIFAR10(
    root=CIFAR10_ROOT, train=False, download=True,
    transform=transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
    ])
)
testloader = torch.utils.data.DataLoader(
    testset, batch_size=256, shuffle=False, num_workers=0
)
labels = np.array(testset.targets)  # (10000,)
num_classes = 10
M = 1000  # CIFAR-10 test set has exactly 1000 samples per class

print(f'Test set: {len(testset)} samples, {num_classes} classes')
print(f'Samples per class: {np.bincount(labels)}')

Test set: 10000 samples, 10 classes
Samples per class: [1000 1000 1000 1000 1000 1000 1000 1000 1000 1000]


In [3]:
# ── Load model checkpoint ─────────────────────────────────────────────────────
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

ckpt = torch.load('data/example_checkpoint.pth', map_location='cpu')

model = build_model('ResNet18', dataset='cifar').to(DEVICE)
model.load_state_dict(ckpt['model'])
model.eval()
print(f"Checkpoint: epoch {ckpt['epoch']}, val acc {ckpt['acc']:.2f}%")

Device: cpu
Checkpoint: epoch 199, val acc 94.87%


In [4]:
# ── Extract features and logits ───────────────────────────────────────────────
_feat_buf = []
handle = model.avgpool.register_forward_hook(
    lambda m, inp, out: _feat_buf.append(torch.flatten(out, 1).detach().cpu())
)
_logit_buf = []
with torch.no_grad():
    for images, _ in testloader:
        _logit_buf.append(model(images.to(DEVICE)).detach().cpu())
handle.remove()

features = torch.cat(_feat_buf).numpy()   # (10000, 512)
logits   = torch.cat(_logit_buf).numpy()  # (10000, 10)
target   = labels

preds   = logits.argmax(axis=1)
correct = (preds == target)

print(f'Features: {features.shape}   Logits: {logits.shape}')
print(f'Test accuracy: {correct.mean()*100:.2f}%')

# Sort by class — stat/geo analyses require contiguous class blocks
sort_idx = np.argsort(target, kind='stable')
X = features[sort_idx]  # class 0 in rows [0,1000), class 1 in [1000,2000), ...

Features: (10000, 512)   Logits: (10000, 10)
Test accuracy: 94.87%


In [5]:
# ── Logit Analysis ───────────────────────────────────────────────────────────
# Operates on all 10000 samples; logit ops are cheap.
results_logit = logit_analysis(logits, correct, target)

print('Logit Analysis:')
for k, v in results_logit.items():
    print(f'  {k:20s}: {v:.4f}')

Logit Analysis:
  auroc               : 0.9313
  confidence          : 0.9805
  entropy             : 0.0539
  atc_score           : 0.9778
  logit_margin        : 9.5435
  energy_score        : -11.5956


In [6]:
# ── Stat Analysis ────────────────────────────────────────────────────────────
# Pairwise-distance/angle ops are O(N^2); subsample to n_sub per class first.
n_sub = 500

rng = np.random.default_rng(42)
sub_idx = np.concatenate([
    rng.choice(M, n_sub, replace=False) + c * M
    for c in range(num_classes)
])
X_sub = X[sub_idx]  # (num_classes * n_sub, 512), still class-ordered

results_stat = stat_analysis(X_sub)

print(f'Stat Analysis  (n_sub={n_sub}/class, N={len(X_sub)}):')
for k, v in results_stat.items():
    print(f'  {k:20s}: {v:.4f}')

Stat Analysis  (n_sub=500/class, N=5000):
  sparsity            : 0.8357
  mean_distance       : 7.2142
  mean_angle          : 1.1163
  mean_covariance     : 0.0120
  numerical_rank      : 512.0000


In [7]:
# ── Geo Analysis (GLUE) ──────────────────────────────────────────────────────
# X_sub rows are class-ordered (n_sub per class), matching geo_analysis convention.
results_geo = geo_analysis(X_sub, num_classes=num_classes, M=n_sub)

print(f'Geo Analysis   (n_sub={n_sub}/class, N={len(X_sub)}):')
for k, v in results_geo.items():
    print(f'  {k:20s}: {v:.4f}')

Geo Analysis   (n_sub=500/class, N=5000):
  participation_ratio : 10.2586
  neural_collapse     : 0.2104
  dimension           : 1.3009
  radius              : 0.5240
  utility             : 0.1900
